# Train a model

Models are plain `nn.Module`s that take packed tensors, so there is no framework to learn: a standard PyTorch training loop works as-is. This notebook writes that loop out in full, step by step, and trains PointNet++ on ModelNet40. The last section shows the same run through the :lightning: [PyTorch Lightning](https://lightning.ai/) wrappers in `torch_pointcloud.lightning`, which trade the explicit loop for less code.

The dataset downloads on first run. `LIMIT_BATCHES` below keeps the default run to a handful of batches so every cell finishes in seconds; set it to `None` for a real run.

In [ ]:
import torch
from torch.utils.data import Subset

import torch_pointcloud as tp
import torch_pointcloud.transforms as T
from torch_pointcloud.utils.data import PointCloudDataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_POINTS = 1024
BATCH_SIZE = 32
EPOCHS = 1
LIMIT_BATCHES = 4

torch.manual_seed(0)
DEVICE

## 1. Data

ModelNet40 ships CAD meshes, so the pipeline samples points on the faces and normalizes the result. The train pipeline adds augmentation on top; the eval pipeline stays deterministic so validation numbers are comparable across epochs.

In [ ]:
from torch_pointcloud.datasets import ModelNet40

sample = T.Compose([
    T.Rescale(keys="pos", method="centroid"),
    T.RandomSampleFaceVertices(keys="pos", face_key="face", num_samples=NUM_POINTS),
    T.KeepItems(keys=["pos", "normal", "label"]),
])
train_tf = T.Compose([
    sample,
    T.RandomScale(keys="pos", scale_range=(0.8, 1.2)),
    T.RandomJitter(keys="pos", sigma=0.01, clip=0.05),
])

train_dataset = ModelNet40(root="data", train=True, transform=train_tf, download=True)
val_dataset = ModelNet40(root="data", train=False, transform=sample, download=True)
len(train_dataset), len(val_dataset)

### Packed batches

Point clouds have different point counts, so batches are *packed* rather than padded: every cloud is concatenated along axis 0 and a `batch` index tags each point with the cloud it came from. `PointCloudDataLoader` is a `DataLoader` with that collation wired in.

`KeepItems` above matters here: without it every batch would also carry the raw mesh faces the points were sampled from, which are far larger than the sample itself.

In [ ]:
if LIMIT_BATCHES is not None:
    n = LIMIT_BATCHES * BATCH_SIZE
    train_dataset = Subset(train_dataset, torch.randperm(len(train_dataset))[:n].tolist())
    val_dataset = Subset(val_dataset, torch.randperm(len(val_dataset))[:n].tolist())

train_loader = PointCloudDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = PointCloudDataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

data = next(iter(train_loader))
{k: tuple(v.shape) for k, v in data.items() if torch.is_tensor(v)}

`pos` is $(N, 3)$ with $N = 32 \times 1024$ points from 32 clouds, `batch` is $(N,)$, and `label` is $(B,)$: one class per cloud.

## 2. Model

`create_model` builds the architecture. `pretrained=False` (the default) gives random weights, which is what training from scratch wants.

In [ ]:
model = tp.create_model("pointnet2-ssg.modelnet40.xu-yan", task="classification").to(DEVICE)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
print("signature: model(x, pos, batch)")
print("in_channels:", model.in_channels, "| num_classes:", model.num_classes)

Classification models take `(x, pos, batch)` and return $(B, C)$ logits. This configuration has `in_channels = 0`, meaning it learns from geometry alone, so `x` is `None`. A model configured with `in_channels = 6` would instead receive `x = torch.cat([pos, normal], dim=1)`.

## 3. Optimizer, scheduler, loss

Nothing point-cloud-specific here: these are the stock `torch.optim` objects.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
criterion = torch.nn.CrossEntropyLoss()

## 4. The training loop

One epoch of training is the familiar five steps, with the only library-specific part being how a batch dict is unpacked onto the device:

1. move `pos`, `batch` and `label` to the device
2. `optimizer.zero_grad()`
3. forward, then loss
4. `loss.backward()`
5. `optimizer.step()`

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, total_correct, total_seen = 0.0, 0, 0

    for data in loader:
        pos = data["pos"].to(device)
        batch = data["batch"].to(device)
        target = data["label"].to(device)

        optimizer.zero_grad()
        logits = model(None, pos, batch)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * target.numel()
        total_correct += int(logits.argmax(dim=1).eq(target).sum())
        total_seen += target.numel()

    return {"loss": total_loss / total_seen, "acc": total_correct / total_seen}

Evaluation is the same walk over the data without the backward pass, under `torch.no_grad()` and with the model in `eval()` mode.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_seen = 0.0, 0, 0

    for data in loader:
        pos = data["pos"].to(device)
        batch = data["batch"].to(device)
        target = data["label"].to(device)

        logits = model(None, pos, batch)
        total_loss += criterion(logits, target).item() * target.numel()
        total_correct += int(logits.argmax(dim=1).eq(target).sum())
        total_seen += target.numel()

    return {"loss": total_loss / total_seen, "acc": total_correct / total_seen}

Driving both from an epoch loop is all that is left. A real run uses `EPOCHS = 200` and the full dataset; the numbers below come from a few batches, so they only prove the plumbing works.

In [ ]:
for epoch in range(EPOCHS):
    train_metrics = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    val_metrics = evaluate(model, val_loader, criterion, DEVICE)
    print(
        f"epoch {epoch + 1}/{EPOCHS}"
        f" | train loss {train_metrics['loss']:.3f} acc {train_metrics['acc']:.3f}"
        f" | val loss {val_metrics['loss']:.3f} acc {val_metrics['acc']:.3f}"
    )

## 5. Save and reload

Weights are a plain `state_dict`. Reload them into a fresh architecture built by the same `create_model` call.

In [ ]:
torch.save(model.state_dict(), "pointnet2_modelnet40.pt")

reloaded = tp.create_model("pointnet2-ssg.modelnet40.xu-yan", task="classification")
reloaded.load_state_dict(torch.load("pointnet2_modelnet40.pt", weights_only=True))
reloaded.eval();

## 6. The same run with Lightning

Everything above is the loop Lightning would otherwise write for you. If you would rather not maintain it, the `lightning` extra (`pip install "torch-pointcloud[lightning]"`) provides two wrappers:

- `PointCloudDataModule` builds the packed loaders from the datasets (forcing `shuffle=True` for train and `False` for val).
- `LitClassificationModel` takes the same model name `create_model` does and owns the loop, the logging and the checkpointing. It maps the batch dict onto `model(x, pos, batch)` through `input_keys`, resolving `x` to `None` when the batch has no point features.

The optimizer and scheduler are passed as **partials**: callables that receive the parameters and the optimizer, so Lightning constructs them at the right moment. `target_key` names the dict key holding the label. The module logs losses; metrics are callbacks, so accuracy comes from a `MetricCallback` wrapping any torchmetrics metric.

`fast_dev_run=True` on the `Trainer` runs a single train and val batch end to end, which catches shape and device bugs in seconds before committing to a full run.

In [ ]:
from functools import partial

import lightning.pytorch as L
from torchmetrics.classification import MulticlassAccuracy

from torch_pointcloud.lightning import LitClassificationModel, MetricCallback, PointCloudDataModule

dm = PointCloudDataModule(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
)
lit = LitClassificationModel(
    "pointnet2-ssg.modelnet40.xu-yan",
    optimizer=partial(torch.optim.AdamW, lr=1e-3, weight_decay=1e-4),
    scheduler=partial(torch.optim.lr_scheduler.CosineAnnealingLR, T_max=200),
    target_key="label",
)

trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    logger=False,
    enable_checkpointing=False,
    callbacks=[MetricCallback(MulticlassAccuracy(num_classes=40), name="acc")],
)
trainer.fit(lit, datamodule=dm)

`trainer.validate` replays the validation loop on its own, printing the same `val/loss` and `val/acc` the plain loop computed by hand.

In [ ]:
trainer.validate(lit, datamodule=dm)

## Beyond this notebook

- **Segmentation.** The loop is identical except the target is per-point: read `data["segment"]` and pass `ignore_index` to the loss. With Lightning, use `LitSegmentationModel` with `target_key="segment"` and optionally `mix_prob` for Mix3D.
- **Longer epochs.** Wrap the train dataset in `RepeatDataset(dataset, loop=k)` to take more optimizer steps per epoch.
- **Layer-wise learning rates.** Build the parameter groups with `torch_pointcloud.utils.optim.generate_param_groups`, or pass `param_groups=...` to the LightningModule.
- **Full recipes.** The :github: [`examples/`](https://github.com/arthurdjn/pytorch-pointcloud/tree/main/examples) directory has end-to-end training and benchmark scripts for most architectures.